# Data Prep from Silver to Gold

Requires silver parquet data

In gold layer we add features (especially spatial features like hexagon) and aggregate data from different datasets, etc.

In [1]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt
import h3
from shapely import from_wkb, from_wkt
from shapely.geometry.base import BaseGeometry
from pathlib import Path
import holidays


from datetime import datetime
import math

from run_config import (
    PATHS,
    RUN_MODE,
    PROCESSED_DIR,
    H3_RESOLUTION,
    GOLD_TIME_UNITS,
    START_DATE,
    END_DATE,
)

BRONZE_CENSUS_TRACTS = PATHS.bronze_census_tracts
BRONZE_COMMUNITY_AREA = PATHS.bronze_community_areas

SILVER_CENSUS_TRACTS = PATHS.silver_census_tracts
SILVER_COMMUNITY_AREA = PATHS.silver_community_areas
BRONZE_POIS = PATHS.bronze_osm_geo

SILVER_TAXI_PATH = PATHS.silver_taxi_trips
SILVER_WEATHER_PATH = PATHS.silver_weatherdata
SILVER_HEXAGON_PATH = PATHS.silver_hexagon

GOLD_TAXI_PATH = PATHS.gold_taxi_trips
GOLD_WEATHER_PATH = PATHS.gold_weatherdata
GOLD_DEMAND_PATHS = {
    "1h": (
        PATHS.gold_1h_demand_hexagon,
        PATHS.gold_1h_demand_census_tracts,
        PATHS.gold_1h_demand_community_areas,
    ),
    "2h": (
        PATHS.gold_2h_demand_hexagon,
        PATHS.gold_2h_demand_census_tracts,
        PATHS.gold_2h_demand_community_areas,
    ),
    "4h": (
        PATHS.gold_4h_demand_hexagon,
        PATHS.gold_4h_demand_census_tracts,
        PATHS.gold_4h_demand_community_areas,
    ),
}
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Effective processing period: intersection of configured and available taxi data
taxi_min_ts, taxi_max_ts = (
    pl.scan_parquet(SILVER_TAXI_PATH)
    .select(
        pl.col("trip_start_timestamp").min().alias("min_ts"),
        pl.col("trip_start_timestamp").max().alias("max_ts"),
    )
    .collect()
    .row(0)
)

if taxi_min_ts is None or taxi_max_ts is None:
    raise ValueError(f"No taxi timestamps found in {SILVER_TAXI_PATH}")

configured_start = datetime.fromisoformat(START_DATE)
configured_end = datetime.fromisoformat(END_DATE)
bounded_start = max(configured_start, taxi_min_ts)
bounded_end = min(configured_end, taxi_max_ts)

if bounded_start > bounded_end:
    raise ValueError(
        "Configured period and available taxi data do not overlap: "
        f"configured={START_DATE}..{END_DATE}, "
        f"available={taxi_min_ts.isoformat()}..{taxi_max_ts.isoformat()}"
    )

START_TS = bounded_start.isoformat()
END_TS = bounded_end.isoformat()

print(f"Run mode: {RUN_MODE}")
print(f"Gold time units: {GOLD_TIME_UNITS}")
print(f"Configured period: {START_DATE} to {END_DATE}")
print(f"Available taxi period: {taxi_min_ts} to {taxi_max_ts}")
print(f"Effective raw processing period: {START_TS} to {END_TS}")
for time_unit in GOLD_TIME_UNITS:
    print(f"Gold {time_unit} outputs: {GOLD_DEMAND_PATHS[time_unit]}")

Run mode: full
Gold time units: ('1h', '2h', '4h')
Configured period: 2024-01-01T00:00:00 to 2026-05-01T00:00:00
Available taxi period: 2024-01-01 00:00:00 to 2026-05-01 00:00:00
Effective raw processing period: 2024-01-01T00:00:00 to 2026-05-01T00:00:00
Gold 1h outputs: (PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_HEXAGON.parquet'), PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_CENSUS_TRACTS.parquet'), PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet'))
Gold 2h outputs: (PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_2H_DEMAND_HEXAGON.parquet'), PosixPath('/Users/moritz/Do

Nachdem in silver alle Duplikate bereinigt wurden, können nun trip_id und taxi_id gedropped werden

## Taxi Data

### Adding hexagon

First we add hexagon data and then compare the hexagons of trips against the city boundaries of chicago. We only want trips with pickup hexagon in the city boundaries

In [2]:
def h3_from_row(row, lat_col: str, lon_col: str, resolution: int):
    """Return an H3 cell or None when either centroid coordinate is missing."""
    lat, lon = row[lat_col], row[lon_col]
    if lat is None or lon is None:
        return None
    return h3.latlng_to_cell(lat, lon, resolution)


def h3_expr(lat_col: str, lon_col: str, resolution: int, alias: str) -> pl.Expr:
    return (
        pl.struct([lat_col, lon_col])
        .map_elements(
            lambda row: h3_from_row(row, lat_col, lon_col, resolution),
            return_dtype=pl.String,
        )
        .alias(alias)
    )


taxi_with_h3 = (
    pl.scan_parquet(SILVER_TAXI_PATH)
    .with_columns([
        h3_expr("pickup_centroid_latitude", "pickup_centroid_longitude", H3_RESOLUTION, "pickup_h3_cell"),
        # dropoff coordinates can be null (~8%), h3_from_row yields None for those
        h3_expr("dropoff_centroid_latitude", "dropoff_centroid_longitude", H3_RESOLUTION, "dropoff_h3_cell"),
    ])
)

# Check if there are trips without hexagon in Chicagos boundaries
valid_chicago_h3_cells = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select("h3_cell")
)

count_before = (
    taxi_with_h3
    .select(pl.len().alias("n_rows_before"))
    .collect()
)

taxi_with_h3_chicago_only = (
    taxi_with_h3
    .join(
        valid_chicago_h3_cells,
        left_on="pickup_h3_cell",
        right_on="h3_cell",
        how="inner",
    )
)

count_after = (
    taxi_with_h3_chicago_only
    .select(pl.len().alias("n_rows_after"))
    .collect()
)

print(count_before)
print(count_after)

dropoff_h3_quality = (
    taxi_with_h3_chicago_only
    .select([
        pl.len().alias("trips"),
        pl.col("dropoff_h3_cell").is_null().sum().alias("missing_dropoff_h3"),
    ])
    .with_columns(
        (100 * pl.col("missing_dropoff_h3") / pl.col("trips")).alias("missing_dropoff_h3_pct")
    )
    .collect()
)
print(dropoff_h3_quality)

/var/folders/dy/nclg91xj2hs9nnr3bq4677q40000gn/T/ipykernel_82341/1663118324.py:54: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  .collect()


shape: (1, 1)
┌───────────────┐
│ n_rows_before │
│ ---           │
│ u32           │
╞═══════════════╡
│ 13387587      │
└───────────────┘
shape: (1, 1)
┌──────────────┐
│ n_rows_after │
│ ---          │
│ u32          │
╞══════════════╡
│ 13387586     │
└──────────────┘
shape: (1, 3)
┌──────────┬────────────────────┬────────────────────────┐
│ trips    ┆ missing_dropoff_h3 ┆ missing_dropoff_h3_pct │
│ ---      ┆ ---                ┆ ---                    │
│ u32      ┆ u32                ┆ f64                    │
╞══════════╪════════════════════╪════════════════════════╡
│ 13387586 ┆ 940428             ┆ 7.024627               │
└──────────┴────────────────────┴────────────────────────┘


### Save gold version


In [3]:
taxi_with_h3_chicago_only.sink_parquet(GOLD_TAXI_PATH)

print(f"Silver taxi parquet written to: {GOLD_TAXI_PATH}")

Silver taxi parquet written to: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/gold_taxi.parquet


## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 24.05.2026

In [4]:
def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    time_unit: str,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create weather features aggregated to time_unit.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations directly to time_unit
    - Create a complete time_unit timestamp spine
    - Reindex to all time buckets
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # The IEM download is requested with tz=America/Chicago. Taxi and weather
    # timestamps therefore both represent local Chicago wall-clock time.
    df["valid"] = pd.to_datetime(df["valid"], errors="coerce")
    df = df.dropna(subset=["valid"])

    # Normalize timezone-aware inputs to local, timezone-naive timestamps.
    if df["valid"].dt.tz is not None:
        df["valid"] = (
            df["valid"].dt.tz_convert("America/Chicago").dt.tz_localize(None)
        )

    start_ts = pd.Timestamp(start_ts)
    end_ts = pd.Timestamp(end_ts)
    if start_ts.tzinfo is not None:
        start_ts = start_ts.tz_convert("America/Chicago").tz_localize(None)
    if end_ts.tzinfo is not None:
        end_ts = end_ts.tz_convert("America/Chicago").tz_localize(None)

    # Keep only observations in the effective taxi period.
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]
    if df.empty:
        raise ValueError(
            f"No weather observations overlap the processing period {start_ts}..{end_ts}"
        )

    # 2. Assign every observation to the requested aggregation bucket.
    df["valid_hour"] = df["valid"].dt.floor(time_unit)

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: mean within the selected time bucket
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: sum within the selected time bucket
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate directly to time_unit (1h or 4h).
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete time_unit index from start to end.
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq=time_unit,
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate continuous numeric values over missing buckets.
    # Missing precipitation must be zero, not an interpolated/extrapolated amount.
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )
    if "p01m" in hourly.columns:
        hourly["p01m"] = hourly["p01m"].fillna(0.0)

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()
    
    # Onehot encoding
    skyc1_dummies = pd.get_dummies(
        gold["skyc1"],
        prefix="skyc1",
        dummy_na=False,
        dtype="int8",
    )

    gold = pd.concat([gold, skyc1_dummies], axis=1)
    
    # Drop unused columns
    gold = gold.drop(columns=["skyc1", "station", "lat", "lon"])

    return gold

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)
print(f"Silver weather rows loaded once: {len(weather_silver):,}")

Silver weather rows loaded once: 23,718


## Points of Interest

In [5]:
pois = gpd.read_parquet(BRONZE_POIS)

print("\nPOIs:")
print(pois.shape)
print(pois.columns)
print(pois.crs)
print(type(pois.geometry.iloc[0]))



POIs:
(9115, 21)
Index(['poi_category', 'name', 'amenity', 'shop', 'railway', 'tourism',
       'historic', 'man_made', 'brand', 'operator', 'opening_hours',
       'addr_housenumber', 'addr_street', 'addr_city', 'website', 'phone',
       'geom_type_original', 'area_m2', 'lat', 'lon', 'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy"

### Hexagon

In [6]:
hexagons = gpd.read_parquet(SILVER_HEXAGON_PATH)

print("Hexagons:")
print(hexagons.shape)
print(hexagons.columns)
print(hexagons.crs)
print(type(hexagons.geometry.iloc[0]))

# CRS prüfen
print("Hexagon CRS:", hexagons.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != hexagons.crs:
    pois = pois.to_crs(hexagons.crs)

print("CRS identisch:", pois.crs == hexagons.crs)

pois_with_h3 = gpd.sjoin(
    pois,
    hexagons[["h3_cell", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_h3.head()
# technische Join-Spalte entfernen
pois_with_h3 = pois_with_h3.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_h3 = pois_with_h3["h3_cell"].isna().sum()

print("POIs gesamt:", len(pois_with_h3))
print("POIs ohne h3_cell:", missing_h3)
print("Anteil ohne h3_cell:", round(missing_h3 / len(pois_with_h3) * 100, 2), "%")

pois_with_h3[["name", "poi_category", "lat", "lon", "h3_cell"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_h3))

duplicate_rows = len(pois_with_h3) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Hexagons:
(853, 3)
Index(['h3_cell', 'h3_resolution', 'geometry'], dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbr

### Census Tract

In [7]:
census_tracts = gpd.read_parquet(SILVER_CENSUS_TRACTS)

print("Census Tracts:")
print(census_tracts.shape)
print(census_tracts.columns)
print(census_tracts.crs)
print(type(census_tracts.geometry.iloc[0]))

# CRS prüfen
print("Census Tract CRS:", census_tracts.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != census_tracts.crs:
    pois = pois.to_crs(census_tracts.crs)

print("CRS identisch:", pois.crs == census_tracts.crs)

pois_with_census_tract = gpd.sjoin(
    pois_with_h3,
    census_tracts[["CENSUS_T_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_census_tract.head()
# technische Join-Spalte entfernen
pois_with_census_tract = pois_with_census_tract.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_census_tract = pois_with_census_tract["CENSUS_T_1"].isna().sum()

print("POIs gesamt:", len(pois_with_census_tract))
print("POIs ohne census tract:", missing_census_tract)
print("Anteil ohne census tract", round(missing_census_tract / len(pois_with_census_tract) * 100, 2), "%")

pois_with_census_tract[["name", "poi_category", "lat", "lon", "CENSUS_T_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_census_tract))

duplicate_rows = len(pois_with_census_tract) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Census Tracts:
(878, 18)
Index(['OBJECTID', 'CENSUS_TRA', 'CENSUS_T_1', 'TRACT_FIPS', 'TRACT_CENT',
       'TRACT_CE_1', 'TRACT_CE_2', 'TRACT_CE_3', 'TRACT_COMM', 'TRACT_NUMA',
       'TRACT_CENS', 'PERIMETER', 'DATA_ADMIN', 'TRACT_CREA', 'TRACT_CR_1',
       'SHAPE_AREA', 'SHAPE_LEN', 'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy":

### Community Area

In [8]:
community_area = gpd.read_parquet(SILVER_COMMUNITY_AREA)

print("Community Area:")
print(community_area.shape)
print(community_area.columns)
print(community_area.crs)
print(type(community_area.geometry.iloc[0]))

# CRS prüfen
print("Community Area CRS:", community_area.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != community_area.crs:
    pois = pois.to_crs(community_area.crs)

print("CRS identisch:", pois.crs == community_area.crs)

pois_with_community_area = gpd.sjoin(
    pois_with_census_tract,
    community_area[["AREA_NUM_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_community_area.head()
# technische Join-Spalte entfernen
pois_with_community_area = pois_with_community_area.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_community_area = pois_with_community_area["AREA_NUM_1"].isna().sum()

print("POIs gesamt:", len(pois_with_community_area))
print("POIs ohne community area:", missing_community_area)
print("Anteil ohne community area", round(missing_community_area / len(pois_with_community_area) * 100, 2), "%")

pois_with_community_area[["name", "poi_category", "lat", "lon", "AREA_NUM_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_community_area))

duplicate_rows = len(pois_with_community_area) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Community Area:
(77, 6)
Index(['AREA_NUMBE', 'COMMUNITY', 'AREA_NUM_1', 'SHAPE_AREA', 'SHAPE_LEN',
       'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north

In [9]:
# Drop all missing h3_cell, CENSUS_T_1 and AREA_NUM_1
pois_with_spatials = pois_with_community_area.dropna(subset=["h3_cell", "CENSUS_T_1", "AREA_NUM_1"])

def group_pois_by_spatial(spatial_col):
    
    poi_h3_categories = pois_with_spatials[[spatial_col, "poi_category"]].copy()

    poi_h3_categories.head()
    
    poi_counts_long = (
    poi_h3_categories
        .groupby([spatial_col, "poi_category"])
        .size()
        .reset_index(name="poi_count")
    )

    poi_counts_long.head()

    poi_counts_wide = (
        poi_counts_long
        .pivot_table(
            index=spatial_col,
            columns="poi_category",
            values="poi_count",
            fill_value=0
        )
        .reset_index()
    )

    poi_counts_wide.head()
    
    poi_counts_wide.columns.name = None

    poi_counts_wide.head()
    
    return poi_counts_wide

pois_grouped_hexagon = group_pois_by_spatial("h3_cell")
pois_grouped_census_tract = group_pois_by_spatial("CENSUS_T_1")
pois_grouped_community_area = group_pois_by_spatial("AREA_NUM_1")

## Hourly Dataset

### Preparations

Here we create an hourly dataset, whereby each hour contains a row for each hexagon. 

We use cyclic encoding for columns like hour, weekday or month to be aware of the distance between time instances (e. g. hour 23 -> 0).

And we merge with the weather data.

These steps are done before the cross join as they are only dependend on time and not on the spatial unit

In [10]:
START = datetime.fromisoformat(START_TS).date()
END = datetime.fromisoformat(END_TS).date()

il_holidays = holidays.country_holidays(
    country="US",
    subdiv="IL", # Illinois = relevant for Chicago
    years=[2024, 2025, 2026]
)

holiday_df = pl.DataFrame(
    [
        {
            "date": d,
            "is_holiday": 1,
        }
        for d, name in sorted(il_holidays.items())
        if START <= d <= END
    ],
    schema={"date": pl.Date, "is_holiday": pl.Int8},
)

In [11]:
def create_temporal_features(time_unit: str) -> pl.LazyFrame:
    """Create the complete local-time feature spine for one interval."""
    effective_start = pd.Timestamp(bounded_start).floor(time_unit)
    effective_end = pd.Timestamp(bounded_end).floor(time_unit)

    weather_gold = create_hourly_weather_gold(
        weather_silver,
        time_unit=time_unit,
        start_ts=effective_start.isoformat(),
        end_ts=effective_end.isoformat(),
    )
    if weather_gold["tmpc"].isna().any():
        raise ValueError(f"Weather contains missing temperatures for {time_unit}")

    hourly_index = pd.date_range(
        start=effective_start,
        end=effective_end,
        freq=time_unit,
        inclusive="both",
        name="datetime_hour",
    )
    hours = (
        pl.from_pandas(pd.DataFrame({"datetime_hour": hourly_index}))
        .with_columns(
            pl.col("datetime_hour")
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
        )
    )

    hours_with_features = (
        hours
        .with_columns([
            pl.col("datetime_hour").dt.month().alias("month"),
            pl.col("datetime_hour").dt.weekday().alias("weekday"),
            pl.col("datetime_hour").dt.hour().alias("hour"),
        ])
        .with_columns([
            ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
            ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),
            ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
            ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
        ])
    )

    weather_features = (
        pl.from_pandas(weather_gold)
        .with_columns(
            pl.col("valid_hour")
            .cast(pl.Datetime("us"))
            .alias("datetime_hour")
        )
        .drop("valid_hour")
        .lazy()
    )

    return (
        hours_with_features.lazy()
        .join(weather_features, on="datetime_hour", how="left")
        .with_columns(pl.col("datetime_hour").dt.date().alias("date"))
        .join(holiday_df.lazy(), on="date", how="left")
        .with_columns(pl.col("is_holiday").fill_null(0).cast(pl.Int8))
    )

### Hexagon Hourly

In [12]:
hexagon = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select([
        pl.col("h3_cell").alias("h3_cell"),  
    ])
    .unique()
    .collect()
).lazy()

hexagon_with_pois = (
    hexagon
    .join(
        pl.from_pandas(pois_grouped_hexagon).lazy(),
        on="h3_cell",
        how="left"
    )
)

### Census Tract Hourly

In [13]:
census_tracts = (
    pl.scan_parquet(SILVER_CENSUS_TRACTS)
    .select([
        pl.col("CENSUS_T_1").alias("census_tract"),  
    ])
    .unique()
    .collect()
).lazy()

census_tracts_with_pois = (
    census_tracts
    .join(
        pl.from_pandas(pois_grouped_census_tract).lazy(),
        left_on="census_tract",
        right_on="CENSUS_T_1",
        how="left"
    )
)

### Community Area Hourly

In [14]:
community_area = (
    pl.scan_parquet(SILVER_COMMUNITY_AREA)
    .select(
        pl.col("AREA_NUM_1").alias("community_area")
    )
    .unique()
    .collect()
).lazy()

community_area_with_pois = (
    community_area
    .join(
        pl.from_pandas(pois_grouped_community_area).lazy(),
        left_on="community_area",
        right_on="AREA_NUM_1",
        how="left"
    )
)

### Function to aggregate taxi data into the spatio-temporal datasets

In [15]:
def add_taxi_data(
    skeleton: pl.LazyFrame | pl.DataFrame,
    taxi: pl.LazyFrame | pl.DataFrame,
    spatial_col: str,
    taxi_spatial_col: str,
    time_unit: str,
    timestamp_col: str = "trip_start_timestamp",
    datetime_col: str = "datetime_hour",
) -> pl.LazyFrame:
    """
    Adds hourly trip-start demand to a complete hour × spatial-unit skeleton.

    Parameters
    ----------
    skeleton:
        Complete base dataset with one row per datetime_col × spatial_col.
        Example: datetime_hour × h3_cell, datetime_hour × census_tract, etc.

    taxi:
        Taxi trip dataset with one row per trip.

    spatial_col:
        Spatial column name in the skeleton/output.
        Example: "h3_cell", "census_tract", "community_area".

    taxi_spatial_col:
        Spatial pickup column name in taxi.
        Example: "pickup_h3_cell", "pickup_census_tract", "pickup_community_area".

    timestamp_col:
        Taxi trip start timestamp column.

    datetime_col:
        Hourly timestamp column used in skeleton/output.

    demand_col:
        Output demand column name.

    Returns
    -------
    pl.LazyFrame
        Skeleton enriched with demand_col.
    """

    if isinstance(skeleton, pl.DataFrame):
        skeleton = skeleton.lazy()

    if isinstance(taxi, pl.DataFrame):
        taxi = taxi.lazy()

    taxi_demand = (
        taxi
        .filter(
            pl.col(timestamp_col).is_not_null()
            & pl.col(taxi_spatial_col).is_not_null() # TODO Census Tract contains nulls
        )
        .with_columns([
            pl.col(timestamp_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
            .alias(datetime_col),

            pl.col(taxi_spatial_col).alias(spatial_col),
        ])
        .group_by([
            datetime_col,
            spatial_col,
        ])
        .agg(
            # aggregations:
            pl.len().alias('trip_count'),
            pl.col('trip_seconds').sum().alias('trip_seconds_sum'),
            pl.col('trip_seconds').mean().alias('trip_seconds_mean'),
            pl.col('trip_seconds').min().alias('trip_seconds_min'),
            pl.col('trip_seconds').max().alias('trip_seconds_max'),
            pl.col('trip_miles').sum().alias('trip_miles_sum'),
            pl.col('trip_miles').mean().alias('trip_miles_mean'),
            pl.col('trip_miles').min().alias('trip_miles_min'),
            pl.col('trip_miles').max().alias('trip_miles_max'),
            pl.col('fare').sum().alias('fare_sum'),
            pl.col('fare').mean().alias('fare_mean'),
            pl.col('fare').min().alias('fare_min'),
            pl.col('fare').max().alias('fare_max'),
            pl.col('tips').sum().alias('tips_sum'),
            pl.col('tips').mean().alias('tips_mean'),
            pl.col('tips').min().alias('tips_min'),
            pl.col('tips').max().alias('tips_max'),
            pl.col('tolls').sum().alias('tolls_sum'),
            pl.col('tolls').mean().alias('tolls_mean'),
            pl.col('tolls').min().alias('tolls_min'),
            pl.col('tolls').max().alias('tolls_max'),
            pl.col('extras').sum().alias('extras_sum'),
            pl.col('extras').mean().alias('extras_mean'),
            pl.col('extras').min().alias('extras_min'),
            pl.col('extras').max().alias('extras_max'),
            pl.col('trip_total').sum().alias('trip_total_sum'),
            pl.col('trip_total').mean().alias('trip_total_mean'),
            pl.col('trip_total').min().alias('trip_total_min'),
            pl.col('trip_total').max().alias('trip_total_max'),
            pl.col('payment_type').drop_nulls().mode().first().alias('most_common_payment_type')
        )    
    )

    result = (
        skeleton
        .with_columns(
            pl.col(datetime_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
            .alias(datetime_col)
        )
        .join(
            taxi_demand,
            on=[datetime_col, spatial_col],
            how="left",
        )
        .with_columns([
            pl.col("trip_count").fill_null(0).alias("trip_count"),

            pl.col("trip_seconds_sum").fill_null(0).alias("trip_seconds_sum"),
            pl.col("trip_seconds_mean").fill_null(0).alias("trip_seconds_mean"),
            pl.col("trip_seconds_min").fill_null(0).alias("trip_seconds_min"),
            pl.col("trip_seconds_max").fill_null(0).alias("trip_seconds_max"),

            pl.col("trip_miles_sum").fill_null(0).alias("trip_miles_sum"),
            pl.col("trip_miles_mean").fill_null(0).alias("trip_miles_mean"),
            pl.col("trip_miles_min").fill_null(0).alias("trip_miles_min"),
            pl.col("trip_miles_max").fill_null(0).alias("trip_miles_max"),

            pl.col("fare_sum").fill_null(0).alias("fare_sum"),
            pl.col("fare_mean").fill_null(0).alias("fare_mean"),
            pl.col("fare_min").fill_null(0).alias("fare_min"),
            pl.col("fare_max").fill_null(0).alias("fare_max"),

            pl.col("tips_sum").fill_null(0).alias("tips_sum"),
            pl.col("tips_mean").fill_null(0).alias("tips_mean"),
            pl.col("tips_min").fill_null(0).alias("tips_min"),
            pl.col("tips_max").fill_null(0).alias("tips_max"),

            pl.col("tolls_sum").fill_null(0).alias("tolls_sum"),
            pl.col("tolls_mean").fill_null(0).alias("tolls_mean"),
            pl.col("tolls_min").fill_null(0).alias("tolls_min"),
            pl.col("tolls_max").fill_null(0).alias("tolls_max"),

            pl.col("extras_sum").fill_null(0).alias("extras_sum"),
            pl.col("extras_mean").fill_null(0).alias("extras_mean"),
            pl.col("extras_min").fill_null(0).alias("extras_min"),
            pl.col("extras_max").fill_null(0).alias("extras_max"),

            pl.col("trip_total_sum").fill_null(0).alias("trip_total_sum"),
            pl.col("trip_total_mean").fill_null(0).alias("trip_total_mean"),
            pl.col("trip_total_min").fill_null(0).alias("trip_total_min"),
            pl.col("trip_total_max").fill_null(0).alias("trip_total_max"),
        ])
        .with_columns(
            pl.col("most_common_payment_type")
            .fill_null("No trips")
            .alias("most_common_payment_type")
        )
    )

    return result

In [16]:
taxi_gold = pl.scan_parquet(GOLD_TAXI_PATH)
generated_outputs = {}

for time_unit in GOLD_TIME_UNITS:
    print(f"\nGenerating {time_unit} Gold demand datasets...")
    temporal_features = create_temporal_features(time_unit)
    hexagon_output, census_output, community_output = GOLD_DEMAND_PATHS[time_unit]

    hexagon_demand = add_taxi_data(
        skeleton=temporal_features.join(hexagon_with_pois, how="cross"),
        taxi=taxi_gold,
        spatial_col="h3_cell",
        taxi_spatial_col="pickup_h3_cell",
        time_unit=time_unit,
    )
    hexagon_demand.sink_parquet(hexagon_output)

    census_demand = add_taxi_data(
        skeleton=(
            temporal_features
            .join(census_tracts_with_pois, how="cross")
            .with_columns(pl.col("census_tract").cast(pl.Int64))
        ),
        taxi=taxi_gold,
        spatial_col="census_tract",
        taxi_spatial_col="pickup_census_tract",
        time_unit=time_unit,
    )
    census_demand.sink_parquet(census_output)

    community_demand = add_taxi_data(
        skeleton=(
            temporal_features
            .join(community_area_with_pois, how="cross")
            .with_columns(pl.col("community_area").cast(pl.Int64))
        ),
        taxi=taxi_gold,
        spatial_col="community_area",
        taxi_spatial_col="pickup_community_area",
        time_unit=time_unit,
    )
    community_demand.sink_parquet(community_output)

    generated_outputs[time_unit] = {
        "hexagon": hexagon_output,
        "census_tract": census_output,
        "community_area": community_output,
    }
    print(f"Finished {time_unit}: {generated_outputs[time_unit]}")


Generating 1h Gold demand datasets...
Finished 1h: {'hexagon': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_HEXAGON.parquet'), 'census_tract': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_CENSUS_TRACTS.parquet'), 'community_area': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet')}

Generating 2h Gold demand datasets...
Finished 2h: {'hexagon': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_2H_DEMAND_HEXAGON.parquet'), 'census_tract': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/processed_data/GOLD_2H

## H3 resolution comparison datasets (spatial-resolution variation)

The main pipeline above varies the **temporal** resolution (`GOLD_TIME_UNITS`) at fixed H3 resolution 8. The assignment additionally asks how patterns and model performance change with **spatial** resolution ("varying hexagon diameter"). The cells below generate the same demand skeletons at H3 resolution 7 (coarser) and 9 (finer).

- The city grid at res 7/9 is derived from the validated res-8 Chicago grid via `cell_to_parent` / `cell_to_children`, so the covered area is identical.
- POI counts are re-aggregated at each resolution from the raw POI coordinates.
- Taxi pickups are re-indexed from pickup coordinates at the requested resolution.
- Outputs: `GOLD_{time}_DEMAND_HEXAGON_R{res}.parquet` in the processed directory.

Generation is gated behind `GENERATE_H3_RESOLUTION_COMPARISON` because the res-9 hourly grid is large. Enable it before running the SVM/NN spatial-resolution experiments.

In [17]:
H3_COMPARISON_RESOLUTIONS = (7, 9)  # res 8 is produced by the main loop above
GENERATE_H3_RESOLUTION_COMPARISON = False  # opt-in: large outputs, see markdown above

POI_FEATURES = ["food_drink", "landmark", "shop", "train_station"]


def make_h3_spatial_context(resolution: int) -> pl.LazyFrame:
    """Chicago H3 grid at the requested resolution with POI counts attached."""
    base_r8 = (
        pl.read_parquet(SILVER_HEXAGON_PATH)
        .get_column("h3_cell").drop_nulls().unique().to_list()
    )
    if resolution == H3_RESOLUTION:
        cells_at_res = sorted(base_r8)
    elif resolution < H3_RESOLUTION:
        cells_at_res = sorted({h3.cell_to_parent(cell, resolution) for cell in base_r8})
    else:
        cells_at_res = sorted({
            child for cell in base_r8 for child in h3.cell_to_children(cell, resolution)
        })

    poi_frame = pois[["lat", "lon", "poi_category"]].dropna(subset=["lat", "lon"]).copy()
    poi_frame["h3_cell"] = [
        h3.latlng_to_cell(lat, lon, resolution)
        for lat, lon in zip(poi_frame["lat"], poi_frame["lon"])
    ]
    poi_counts = (
        poi_frame.groupby(["h3_cell", "poi_category"]).size().unstack(fill_value=0).reset_index()
    )
    for col in POI_FEATURES:
        if col not in poi_counts:
            poi_counts[col] = 0

    return (
        pl.DataFrame({"h3_cell": cells_at_res}).lazy()
        .join(pl.from_pandas(poi_counts[["h3_cell", *POI_FEATURES]]).lazy(), on="h3_cell", how="left")
        .with_columns([pl.col(col).fill_null(0) for col in POI_FEATURES])
        .with_columns(pl.lit(resolution).cast(pl.Int8).alias("h3_resolution"))
    )

In [18]:
if GENERATE_H3_RESOLUTION_COMPARISON:
    resolution_outputs = {}
    for resolution in H3_COMPARISON_RESOLUTIONS:
        spatial_context = make_h3_spatial_context(resolution)
        taxi_col = f"pickup_h3_r{resolution}"
        taxi_at_res = pl.scan_parquet(GOLD_TAXI_PATH).with_columns(
            h3_expr("pickup_centroid_latitude", "pickup_centroid_longitude", resolution, taxi_col)
        )
        for time_unit in GOLD_TIME_UNITS:
            temporal_features = create_temporal_features(time_unit)
            output_path = PROCESSED_DIR / f"GOLD_{time_unit.upper()}_DEMAND_HEXAGON_R{resolution}.parquet"
            demand = add_taxi_data(
                skeleton=temporal_features.join(spatial_context, how="cross"),
                taxi=taxi_at_res,
                spatial_col="h3_cell",
                taxi_spatial_col=taxi_col,
                time_unit=time_unit,
            )
            print(f"Writing {output_path.name} ...")
            demand.sink_parquet(output_path)
            resolution_outputs[(resolution, time_unit)] = output_path

    for (resolution, time_unit), output_path in resolution_outputs.items():
        result = duckdb.sql(f"""
            SELECT COUNT(*) AS rows, SUM(trip_count) AS trips
            FROM read_parquet('{output_path}')
        """).fetchone()
        print(f"r{resolution} {time_unit}: {output_path.name} -> rows={result[0]:,}, trips={result[1]:,}")
else:
    print("Set GENERATE_H3_RESOLUTION_COMPARISON = True to build the res-7/res-9 datasets.")

Writing GOLD_1H_DEMAND_HEXAGON_R7.parquet ...
Writing GOLD_2H_DEMAND_HEXAGON_R7.parquet ...
Writing GOLD_4H_DEMAND_HEXAGON_R7.parquet ...
Writing GOLD_1H_DEMAND_HEXAGON_R9.parquet ...
Writing GOLD_2H_DEMAND_HEXAGON_R9.parquet ...
Writing GOLD_4H_DEMAND_HEXAGON_R9.parquet ...
r7 1h: GOLD_1H_DEMAND_HEXAGON_R7.parquet -> rows=3,104,600, trips=13,387,571
r7 2h: GOLD_2H_DEMAND_HEXAGON_R7.parquet -> rows=1,552,376, trips=13,387,571
r7 4h: GOLD_4H_DEMAND_HEXAGON_R7.parquet -> rows=776,264, trips=13,387,571
r9 1h: GOLD_1H_DEMAND_HEXAGON_R9.parquet -> rows=121,957,675, trips=13,387,571
r9 2h: GOLD_2H_DEMAND_HEXAGON_R9.parquet -> rows=60,981,823, trips=13,387,571
r9 4h: GOLD_4H_DEMAND_HEXAGON_R9.parquet -> rows=30,493,897, trips=13,387,571


In [19]:
for time_unit, outputs in generated_outputs.items():
    for spatial_unit, output_path in outputs.items():
        result = duckdb.sql(f"""
            SELECT COUNT(*) AS rows, SUM(trip_count) AS trips
            FROM read_parquet('{output_path}')
        """).fetchone()
        print(time_unit, spatial_unit, output_path.name, result)

1h hexagon GOLD_1H_DEMAND_HEXAGON.parquet (17422525, 13387586)
1h census_tract GOLD_1H_DEMAND_CENSUS_TRACTS.parquet (17933150, 2830710)
1h community_area GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet (1572725, 13387586)
2h hexagon GOLD_2H_DEMAND_HEXAGON.parquet (8711689, 13387586)
2h census_tract GOLD_2H_DEMAND_CENSUS_TRACTS.parquet (8967014, 2830710)
2h community_area GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet (786401, 13387586)
4h hexagon GOLD_4H_DEMAND_HEXAGON.parquet (4356271, 13387586)
4h census_tract GOLD_4H_DEMAND_CENSUS_TRACTS.parquet (4483946, 2830710)
4h community_area GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet (393239, 13387586)


## Quality Check

In [20]:
con = duckdb.connect()

for time_unit, output_path in (
    (unit, outputs["community_area"])
    for unit, outputs in generated_outputs.items()
):
    columns = con.sql(f"""
        DESCRIBE SELECT * FROM read_parquet('{output_path}')
    """).df()["column_name"].tolist()
    parts = [
        f"""
        SELECT '{col}' AS column_name, COUNT(*) AS n_rows,
               SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_nulls,
               ROUND(100.0 * SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS null_pct
        FROM read_parquet('{output_path}')
        """
        for col in columns
    ]
    query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"
    print(f"\nNull report for {time_unit}: {output_path.name}")
    print(con.sql(query).df().to_string(index=False))


Null report for 1h: GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet
             column_name  n_rows  n_nulls  null_pct
                 weekday 1572725      0.0       0.0
               tolls_max 1572725      0.0       0.0
           datetime_hour 1572725      0.0       0.0
                   month 1572725      0.0       0.0
               skyc1_BKN 1572725      0.0       0.0
               month_cos 1572725      0.0       0.0
               skyc1_CLR 1572725      0.0       0.0
                    p01m 1572725      0.0       0.0
                hour_sin 1572725      0.0       0.0
                    sknt 1572725      0.0       0.0
                hour_cos 1572725      0.0       0.0
             weekday_sin 1572725      0.0       0.0
                    relh 1572725      0.0       0.0
               skyc1_VV  1572725      0.0       0.0
                    hour 1572725      0.0       0.0
             weekday_cos 1572725      0.0       0.0
                    date 1572725      0.0       0.0
    